In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import pyspark.sql.functions as F

from lakehouse_common import create_spark
spark = create_spark("Delivery - 04_fact_and_ml_gold")

# 1. Đọc Silver + Dim
df_delivery = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_silver")
df_courier = spark.read.format("delta").load("s3a://lakehouse/warehouse/dim_courier_silver")

print(f"Silver: {df_delivery.count()} dòng")

# Keep order region and courier region as separate attributes.
from lakehouse_common import join_courier_history
fact_delivery = join_courier_history(df_delivery, df_courier)
fact_count = fact_delivery.count()
if fact_count == 0:
    raise ValueError("Gold fact is empty")

# 5. Ghi Gold
fact_delivery.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("s3a://lakehouse/warehouse/fact_delivery_task_gold")
print("Đã lưu bảng Fact Gold thành công!")

# 6. Train ML dự báo ETA
ml_data = fact_delivery.filter((F.col("distance_km") > 0) & (F.col("delivery_minutes") > 0))
ml_count = ml_data.count()
print(f"Số dòng dùng để train ML: {ml_count}")
if ml_count == 0:
    raise ValueError("ml_data rỗng - kiểm tra distance_km và delivery_minutes")

assembler = VectorAssembler(inputCols=["distance_km"], outputCol="features")
ml_data = assembler.transform(ml_data)

import os
cutoff = os.getenv("TRAIN_END", "2022-10-01")
train_data = ml_data.filter(F.col("order_date") < cutoff)
test_data = ml_data.filter(F.col("order_date") >= cutoff)
if train_data.count() < 2 or test_data.count() < 2:
    raise ValueError("Need at least two rows in train and test; adjust TRAIN_END")

lr = LinearRegression(featuresCol="features", labelCol="delivery_minutes")
lr_model = lr.fit(train_data)

predictions = lr_model.transform(test_data)
rmse = RegressionEvaluator(labelCol="delivery_minutes", predictionCol="prediction", metricName="rmse").evaluate(predictions)
r2 = RegressionEvaluator(labelCol="delivery_minutes", predictionCol="prediction", metricName="r2").evaluate(predictions)

print("-" * 50)
print("KẾT QUẢ HUẤN LUYỆN MÔ HÌNH DỰ BÁO ETA")
print(f"- RMSE: {rmse:.2f} phút")
print(f"- R2: {r2:.3f}")
print(f"- Hệ số (Coefficient): {lr_model.coefficients[0]:.2f}")
print(f"- Intercept: {lr_model.intercept:.2f}")
print("-" * 50)

predictions.select("distance_km", "delivery_minutes",
                   F.round("prediction", 2).alias("predicted_minutes")).show(5)

In [ ]:
dim = spark.read.format("delta").load("s3a://lakehouse/warehouse/dim_courier_silver")
silver = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_silver") \
              .withColumn("order_date", F.to_date("accept_ts"))

dim.agg(F.min("effective_date"), F.max("effective_date"), F.countDistinct("courier_id")).show()
silver.agg(F.min("order_date"), F.max("order_date"), F.countDistinct("courier_id")).show()

In [ ]:
fact = spark.read.format("delta").load("s3a://lakehouse/warehouse/fact_delivery_task_gold")
print(fact.stat.corr("distance_km", "delivery_minutes"))
fact.select("delivery_minutes").summary("min", "25%", "50%", "75%", "max").show()

In [ ]:
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import pyspark.sql.functions as F

s = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_silver") \
         .withColumn("order_date", F.to_date("accept_ts"))

w = Window.partitionBy("courier_id", "order_date").orderBy("accept_ts", "order_id") \
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)

feat = s \
    .withColumn("accept_hour", F.hour("accept_ts") + F.minute("accept_ts") / 60) \
    .withColumn("dow", F.dayofweek("order_date")) \
    .withColumn("order_rank", F.count("*").over(w)) \
    .withColumn("cum_avg_dist", F.avg("distance_km").over(w)) \
    .withColumn("cum_max_dist", F.max("distance_km").over(w)) \
    .withColumn("region_str", F.col("region_id").cast("string")) \
    .withColumn("label_log", F.log1p("delivery_minutes"))

train = feat.filter(F.col("order_date") < "2022-10-01")
test  = feat.filter(F.col("order_date") >= "2022-10-01")
if train.count() < 2 or test.count() < 2:
    raise ValueError("Empty or too small chronological split")
print(f"train={train.count()}, test={test.count()}")

idx_city   = StringIndexer(inputCol="city", outputCol="city_idx", handleInvalid="keep")
idx_aoi    = StringIndexer(inputCol="aoi_type", outputCol="aoi_type_idx", handleInvalid="keep")
idx_region = StringIndexer(inputCol="region_str", outputCol="region_idx", handleInvalid="keep")

cols = ["distance_km", "accept_hour", "dow", "order_rank",
        "cum_avg_dist", "cum_max_dist", "region_idx", "aoi_type_idx"]
assembler = VectorAssembler(inputCols=cols, outputCol="features", handleInvalid="error")

gbt = GBTRegressor(featuresCol="features", labelCol="label_log",
                   maxIter=80, maxDepth=6, maxBins=256, seed=42)

model = Pipeline(stages=[idx_city, idx_aoi, idx_region, assembler, gbt]).fit(train)
pred = model.transform(test).withColumn("pred_min", F.greatest(F.expm1("prediction"), F.lit(0.0)))

def ev(label, predcol, metric):
    return round(RegressionEvaluator(labelCol=label, predictionCol=predcol, metricName=metric).evaluate(pred), 3)

print("MODEL    rmse", ev("delivery_minutes", "pred_min", "rmse"),
      "| mae", ev("delivery_minutes", "pred_min", "mae"),
      "| r2",  ev("delivery_minutes", "pred_min", "r2"))

med = train.approxQuantile("delivery_minutes", [0.5], 0.01)[0]
mean_train = train.agg(F.avg("delivery_minutes")).first()[0]
base = test.withColumn("p_med", F.lit(med)).withColumn("p_mean", F.lit(mean_train))
for name, c in [("median", "p_med"), ("mean", "p_mean")]:
    print(f"BASELINE {name:6s} rmse",
          round(RegressionEvaluator(labelCol="delivery_minutes", predictionCol=c, metricName="rmse").evaluate(base), 3),
          "| mae",
          round(RegressionEvaluator(labelCol="delivery_minutes", predictionCol=c, metricName="mae").evaluate(base), 3))

for n, v in sorted(zip(cols, model.stages[-1].featureImportances.toArray()), key=lambda x: -x[1]):
    print(f"{n:16s} {v:.3f}")

In [ ]:
f = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_silver") \
         .filter(F.col("delivery_minutes") <= 1440)
print("pearson :", f.stat.corr("distance_km", "delivery_minutes"))
print("log-log :", f.select(F.log1p("distance_km").alias("x"), F.log1p("delivery_minutes").alias("y")).stat.corr("x", "y"))

# Baseline MAE trên tập test (cần train/test từ cell trước)
med = train.approxQuantile("delivery_minutes", [0.5], 0.01)[0]
print("baseline MAE (trung bình):", test.select(F.avg(F.abs(F.col("delivery_minutes") - mean_train))).first()[0])
print("baseline MAE (trung vị)  :", test.select(F.avg(F.abs(F.col("delivery_minutes") - med))).first()[0])

In [ ]:
f = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_silver") \
         .filter(F.col("delivery_minutes") <= 1440)

f.select("distance_km").summary("min", "25%", "50%", "75%", "95%", "99%", "max").show()

for cap in [10, 20, 50]:
    g = f.filter(F.col("distance_km") <= cap)
    print(f"distance <= {cap} km: n={g.count()}, pearson={g.stat.corr('distance_km', 'delivery_minutes'):.4f}")